# 01 - Dataset Analysis

This notebook inspects `dataset/raw/phishing.csv` without assuming its schema. It produces reproducible quality reports and exploratory figures. No final model is trained here.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_analysis import (
    infer_target_column, high_correlation_pairs, random_forest_importance,
    save_exploratory_figures, save_overview_reports,
)
from src.data_preprocessing import unseen_domain_status

raw_path = PROJECT_ROOT / 'dataset' / 'raw' / 'phishing.csv'
results_dir = PROJECT_ROOT / 'results'
if not raw_path.exists():
    raise FileNotFoundError(f'Place the dataset at {raw_path}')
df = pd.read_csv(raw_path)
target_column = infer_target_column(df)
print(f'Shape: {df.shape}')
print(f'Target column: {target_column}')
df.head()

In [ ]:
print('Column names:', list(df.columns))
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().rename('missing_count').to_frame())
print('Duplicate rows:', int(df.duplicated().sum()))
display(df.nunique(dropna=False).rename('unique_count').to_frame())
display(df[target_column].value_counts(dropna=False).rename('count').to_frame())
display(df.describe(include='all').transpose())

In [ ]:
reports = save_overview_reports(df, target_column, results_dir)
save_exploratory_figures(df, target_column, results_dir)
for name, report in reports.items():
    print(name)
    display(report.head())
print('Saved reports and figures to:', results_dir)

In [ ]:
print('Unseen-domain evaluation status:')
print(unseen_domain_status(df))
print('Interpretation: this CSV contains encoded URL-related features but no URL/domain identifier.')

## Phase 2 checkpoint

Completed: schema inspection, quality checks, class distribution, descriptive statistics, feature information, and exploratory figures. The dataset has no missing or duplicate rows in the inspected file.

Important limitation: a domain-disjoint or unseen-domain evaluation cannot be performed from this CSV because it has no URL, domain, host, or website identifier. A later dataset revision must include such an identifier.